# Week 2 Homework: Complete Feature Engineering and Baseline Models

In Session 1, you implemented **6 features** using test-driven development. Now you'll implement **8 more features** to complete the feature set.

---

## Overview

**Your Task**: Implement 8 features using the TDD approach from Session 1.

### Features You Will Implement (8 features)

| # | Feature | Type | Skills Practiced |
|---|---------|------|------------------|
| 1 | `days_since_last_encounter` | Temporal | Simple date difference |
| 2 | `days_since_medication_change` | Temporal | Medication timeline |
| 3 | `emergency_visits_last_180d` | Rolling Window | Event counting |
| 4 | `medication_changes_last_90d` | Rolling Window | Change detection |
| 5 | `active_medication_count` | Clinical State | Active item logic |
| 6 | `hba1c_trend_last_180d` | Trend | Direction (-1/0/1) |
| 7 | `longest_care_gap_days` | Gap Detection | Max of consecutive gaps |
| 8 | `bmi_category` | Categorical | BMI calculation, binning |

### Features Provided to You (8 features)

| # | Feature | Type |
|---|---------|------|
| 1 | `days_since_emergency_visit` | Temporal |
| 2 | `days_since_hospitalization` | Temporal |
| 3 | `hospitalizations_last_365d` | Rolling Window |
| 4 | `diabetes_complication_count` | Clinical State |
| 5 | `care_gaps_count` | Gap Detection |
| 6 | `current_diastolic_bp` | Clinical (LOINC 8462-4) |
| 7 | `bp_trend_last_180d` | Trend |
| 8 | `egfr_trend_last_365d` | Trend |

**Total**: 6 (Session 1) + 8 (your work) + 8 (provided) = **22 features**

---

## Deliverables

1. Implement all 8 features with the 3-step TDD process
2. Verify all features against TRUE values
3. Transfer your functions to ExtendedPatientProfile class methods
4. Generate updated classifier training data with 22 features
5. Compare baseline performance: 6 features vs 22 features

---

## Setup

## Setting Environment Up for Colab

Mount Google Drive and set the repository path so this notebook can access the EHR data and source code.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

### Project Configuration

This notebook loads paths from a YAML config file instead of hard-coded paths.

**Setup (one-time):**
1. In Colab's left sidebar, click the **Key** icon (Secrets)
2. Add a secret named `PROJECT_CONFIG_PATH`
3. Set the value to your config file path (e.g., `/content/drive/MyDrive/Project/config.yaml`)
4. Toggle "Notebook access" ON

In [ ]:
from google.colab import userdata
import yaml

try:
    config_path = userdata.get('PROJECT_CONFIG_PATH')
except:
    config_path = input("Enter path to your config.yaml: ")

with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

REPO_PATH = config['repo_path']
print(f"Repository path: {REPO_PATH}")

### Install Source Code as Package

`pip install` installs the local source code as a Python package so you can import directly from it (e.g., `from week_2.helpers import get_col`). Re-run this cell after making changes to the source code.

In [ ]:
!pip install {REPO_PATH}/src_solutions/ -q

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os

from week_2.helpers import get_col
DATA_DIR = os.path.join(REPO_PATH, "data/week_1/processed_data/csv")

print("Libraries loaded")

In [ ]:
# Load all data tables
patients = pd.read_csv(os.path.join(DATA_DIR, "patients.csv"))
encounters = pd.read_csv(os.path.join(DATA_DIR, "encounters.csv"), low_memory=False)
conditions = pd.read_csv(os.path.join(DATA_DIR, "conditions.csv"))
medications = pd.read_csv(os.path.join(DATA_DIR, "medications.csv"))
observations = pd.read_csv(os.path.join(DATA_DIR, "observations.csv"), low_memory=False)

# Set up column references
patient_id_col = get_col(patients, 'id')
birthdate_col = get_col(patients, 'birthdate')
enc_patient_col = get_col(encounters, 'patient')
enc_date_col = get_col(encounters, 'date')
enc_class_col = get_col(encounters, 'encounterclass')
enc_desc_col = get_col(encounters, 'description')
obs_patient_col = get_col(observations, 'patient')
obs_desc_col = get_col(observations, 'description')
obs_value_col = get_col(observations, 'value')
obs_date_col = get_col(observations, 'date')
obs_code_col = get_col(observations, 'code')
med_patient_col = get_col(medications, 'patient')
med_start_col = get_col(medications, 'start')
med_stop_col = get_col(medications, 'stop')
cond_patient_col = get_col(conditions, 'patient')
cond_desc_col = get_col(conditions, 'description')

print(f"Loaded: {len(patients):,} patients, {len(encounters):,} encounters")

In [ ]:
# Classify encounters by type
def classify_encounter(description):
    desc = str(description).lower()
    if 'emergency' in desc:
        return 'emergency'
    elif any(kw in desc for kw in ['inpatient', 'admission to', 'hospital admission']):
        return 'inpatient'
    elif 'outpatient' in desc:
        return 'outpatient'
    else:
        return 'ambulatory'

encounters['ENCOUNTER_CLASS'] = encounters[enc_desc_col].apply(classify_encounter)
print("Encounter classification complete")

### Session 1 Features (Already Implemented)

These 6 features are from Session 1:
- `days_since_last_hba1c`
- `current_hba1c_level`
- `encounters_last_90d`
- `age_at_date`
- `current_systolic_bp`
- `current_egfr`

In [ ]:
# Session 1 features (copy from session)
def days_since_last_hba1c(patient_id, cutoff_date):
    """Days since most recent HbA1c reading before cutoff."""
    patient_obs = observations[observations[obs_patient_col] == patient_id].copy()
    patient_obs['date'] = pd.to_datetime(patient_obs[obs_date_col], utc=True)
    hba1c_before = patient_obs[
        (patient_obs['date'] <= cutoff_date) &
        (patient_obs[obs_desc_col].str.contains('hemoglobin a1c|hba1c', case=False, na=False))
    ]
    if len(hba1c_before) == 0:
        return None
    last_date = hba1c_before['date'].max()
    return (cutoff_date - last_date).days

def current_hba1c_level(patient_id, cutoff_date):
    """Most recent HbA1c value before cutoff."""
    patient_obs = observations[observations[obs_patient_col] == patient_id].copy()
    patient_obs['date'] = pd.to_datetime(patient_obs[obs_date_col], utc=True)
    hba1c_before = patient_obs[
        (patient_obs['date'] <= cutoff_date) &
        (patient_obs[obs_desc_col].str.contains('hemoglobin a1c|hba1c', case=False, na=False))
    ]
    if len(hba1c_before) == 0:
        return None
    last_value = hba1c_before.loc[hba1c_before['date'].idxmax(), obs_value_col]
    return float(last_value)

def encounters_last_90d(patient_id, cutoff_date):
    """Count encounters in 90 days before cutoff."""
    patient_enc = encounters[encounters[enc_patient_col] == patient_id].copy()
    patient_enc['date'] = pd.to_datetime(patient_enc[enc_date_col], errors='coerce', utc=True)
    if cutoff_date.tzinfo is None:
        cutoff_date = pd.to_datetime(cutoff_date, utc=True)
    window_start = cutoff_date - timedelta(days=90)
    enc_in_window = patient_enc[
        (patient_enc['date'] >= window_start) & 
        (patient_enc['date'] <= cutoff_date)
    ]
    return len(enc_in_window)

def age_at_date(patient_id, cutoff_date):
    """Patient's age in years at cutoff date."""
    patient_info = patients[patients[patient_id_col] == patient_id].iloc[0]
    birthdate = pd.to_datetime(patient_info[birthdate_col])
    if isinstance(cutoff_date, str):
        cutoff_date = pd.to_datetime(cutoff_date)
    age = cutoff_date.year - birthdate.year
    if (cutoff_date.month, cutoff_date.day) < (birthdate.month, birthdate.day):
        age -= 1
    return age

def current_systolic_bp(patient_id, cutoff_date):
    """Most recent systolic BP value before cutoff. LOINC: 8480-6"""
    LOINC_SYSTOLIC_BP = '8480-6'
    patient_obs = observations[observations[obs_patient_col] == patient_id].copy()
    patient_obs['date'] = pd.to_datetime(patient_obs[obs_date_col], utc=True)
    bp_before = patient_obs[
        (patient_obs['date'] <= cutoff_date) &
        ((patient_obs[obs_code_col] == LOINC_SYSTOLIC_BP) |
         (patient_obs[obs_desc_col].str.lower().str.contains('systolic', na=False)))
    ]
    if len(bp_before) == 0:
        return None
    last_value = bp_before.loc[bp_before['date'].idxmax(), obs_value_col]
    return float(last_value)

def current_egfr(patient_id, cutoff_date):
    """Most recent eGFR value before cutoff. LOINC: 33914-3"""
    LOINC_EGFR = '33914-3'
    patient_obs = observations[observations[obs_patient_col] == patient_id].copy()
    patient_obs['date'] = pd.to_datetime(patient_obs[obs_date_col], utc=True)
    egfr_before = patient_obs[
        (patient_obs['date'] <= cutoff_date) &
        ((patient_obs[obs_code_col] == LOINC_EGFR) |
         (patient_obs[obs_desc_col].str.lower().str.contains('glomerular|egfr', na=False)))
    ]
    if len(egfr_before) == 0:
        return None
    last_value = egfr_before.loc[egfr_before['date'].idxmax(), obs_value_col]
    return float(last_value)

print("Session 1 features loaded")

### Select Sample Patient

In [ ]:
# Use a pre-selected patient with rich, diverse data for TDD
# This patient was chosen because they have:
# - Multiple emergency visits AND hospitalizations (for days_since features)
# - Medication changes (for medication_changes_last_90d)
# - HbA1c and BP readings with significant trends (for trend features)
# - A cutoff date that produces non-trivial values for 20/22 features

SAMPLE_PATIENT_ID = "21a69c84-ff08-4928-9e96-95c8aac45a6f"
CUTOFF_DATE = pd.to_datetime("2012-07-05", utc=True)

# Set up patient data
patient_enc = encounters[encounters[enc_patient_col] == SAMPLE_PATIENT_ID].copy()
patient_enc['date'] = pd.to_datetime(patient_enc[enc_date_col], errors='coerce', utc=True)
patient_enc = patient_enc.sort_values('date')

# Get patient observations
patient_obs = observations[observations[obs_patient_col] == SAMPLE_PATIENT_ID].copy()
patient_obs['date'] = pd.to_datetime(patient_obs[obs_date_col], utc=True)

# Get patient medications
patient_meds = medications[medications[med_patient_col] == SAMPLE_PATIENT_ID].copy()
patient_meds['start_date'] = pd.to_datetime(patient_meds[med_start_col], errors='coerce', utc=True)
if med_stop_col and med_stop_col in patient_meds.columns:
    patient_meds['stop_date'] = pd.to_datetime(patient_meds[med_stop_col], errors='coerce', utc=True)

# Show patient summary
n_enc = len(patient_enc)
n_hba1c = len(patient_obs[patient_obs[obs_desc_col].str.contains('hemoglobin a1c|hba1c', case=False, na=False)])
n_meds = len(patient_meds)
n_emergency = len(patient_enc[patient_enc[enc_desc_col].str.lower().str.contains('emergency', na=False)])
n_hosp = len(patient_enc[patient_enc[enc_desc_col].str.lower().str.contains('inpatient|admission to|hospital admission', na=False)])

print(f"Sample patient: {SAMPLE_PATIENT_ID}")
print(f"  Encounters: {n_enc}")
print(f"  HbA1c readings: {n_hba1c}")
print(f"  Medications: {n_meds}")
print(f"  Emergency visits: {n_emergency}")
print(f"  Hospitalizations: {n_hosp}")
print(f"Cutoff date: {CUTOFF_DATE.date()}")

---

# Part 1: Features You Implement (8 features)

Use the **3-step TDD process** for each feature:
1. **Step 1**: Manually compute the TRUE value
2. **Step 2**: Implement the function
3. **Step 3**: Verify computed == TRUE

---

## Feature 1: `days_since_last_encounter`

**Definition**: Days since the most recent encounter (of any type) before the cutoff date.

**Clinical Relevance**: 
- Measures recency of care engagement
- Longer gaps may indicate disengagement or barriers to care

**Pattern Type**: Temporal (simple date difference)

### Step 1: Manually Compute TRUE Value

In [ ]:
# STEP 1: Manually compute TRUE value
enc_before_cutoff = patient_enc[patient_enc['date'] <= CUTOFF_DATE].sort_values('date')

print(f"Encounters BEFORE cutoff ({CUTOFF_DATE.date()}):")
for _, row in enc_before_cutoff.tail(5).iterrows():
    print(f"  {row['date'].date()}")

if len(enc_before_cutoff) > 0:
    last_enc_date = enc_before_cutoff['date'].max()
    TRUE_days_since_last_encounter = (CUTOFF_DATE - last_enc_date).days
    print(f"\nLast encounter: {last_enc_date.date()}")
    print(f"TRUE VALUE: {TRUE_days_since_last_encounter} days")
else:
    TRUE_days_since_last_encounter = None
    print("\nNo encounters before cutoff - TRUE VALUE: None")

### Step 2: Implement the Function

In [ ]:
# STEP 2: Implement the function
def days_since_last_encounter(patient_id, cutoff_date):
    """
    Days since most recent encounter before cutoff.
    
    Args:
        patient_id: Patient identifier
        cutoff_date: Date to calculate from
    
    Returns:
        int: Days since last encounter, or None if no encounters
    """
    patient_enc = encounters[encounters[enc_patient_col] == patient_id].copy()
    patient_enc['date'] = pd.to_datetime(patient_enc[enc_date_col], errors='coerce', utc=True)
    
    enc_before = patient_enc[patient_enc['date'] <= cutoff_date]
    
    if len(enc_before) == 0:
        return None
    
    last_date = enc_before['date'].max()
    return (cutoff_date - last_date).days

print("days_since_last_encounter function defined")

### Step 3: Verify Against TRUE Value

In [ ]:
# STEP 3: Verify
computed = days_since_last_encounter(SAMPLE_PATIENT_ID, CUTOFF_DATE)
print(f"TRUE value:     {TRUE_days_since_last_encounter} days")
print(f"COMPUTED value: {computed} days")

if computed == TRUE_days_since_last_encounter:
    print("\nMATCH! Feature implementation is correct.")
else:
    print("\nMISMATCH! There's a bug in the implementation.")

---

## Feature 2: `days_since_medication_change`

**Definition**: Days since the most recent medication start or stop before the cutoff date.

**Clinical Relevance**:
- Recent medication changes may affect patient stability
- Important for monitoring treatment adjustments

**Pattern Type**: Temporal (medication timeline)

### Step 1: Manually Compute TRUE Value

In [ ]:
# STEP 1: Manually compute TRUE value
# Get all medication changes (starts and stops) before cutoff
med_changes = []

for _, row in patient_meds.iterrows():
    if pd.notna(row['start_date']) and row['start_date'] <= CUTOFF_DATE:
        med_changes.append(row['start_date'])
    if 'stop_date' in row and pd.notna(row.get('stop_date')) and row['stop_date'] <= CUTOFF_DATE:
        med_changes.append(row['stop_date'])

print(f"Medication changes BEFORE cutoff ({CUTOFF_DATE.date()}):")
med_changes_sorted = sorted(med_changes)
for d in med_changes_sorted[-5:]:
    print(f"  {d.date()}")

if med_changes:
    last_change_date = max(med_changes)
    TRUE_days_since_medication_change = (CUTOFF_DATE - last_change_date).days
    print(f"\nLast medication change: {last_change_date.date()}")
    print(f"TRUE VALUE: {TRUE_days_since_medication_change} days")
else:
    TRUE_days_since_medication_change = None
    print("\nNo medication changes - TRUE VALUE: None")

### Step 2: Implement the Function

In [ ]:
# STEP 2: Implement the function
def days_since_medication_change(patient_id, cutoff_date):
    """
    Days since most recent medication change (start or stop) before cutoff.
    
    Args:
        patient_id: Patient identifier
        cutoff_date: Date to calculate from
    
    Returns:
        int: Days since last medication change, or None if no changes
    """
    patient_meds = medications[medications[med_patient_col] == patient_id].copy()
    patient_meds['start_date'] = pd.to_datetime(patient_meds[med_start_col], errors='coerce', utc=True)
    
    med_changes = []
    
    for _, row in patient_meds.iterrows():
        if pd.notna(row['start_date']) and row['start_date'] <= cutoff_date:
            med_changes.append(row['start_date'])
        if med_stop_col and med_stop_col in patient_meds.columns:
            stop_date = pd.to_datetime(row[med_stop_col], errors='coerce', utc=True)
            if pd.notna(stop_date) and stop_date <= cutoff_date:
                med_changes.append(stop_date)
    
    if not med_changes:
        return None
    
    last_change = max(med_changes)
    return (cutoff_date - last_change).days

print("days_since_medication_change function defined")

### Step 3: Verify Against TRUE Value

In [ ]:
# STEP 3: Verify
computed = days_since_medication_change(SAMPLE_PATIENT_ID, CUTOFF_DATE)
print(f"TRUE value:     {TRUE_days_since_medication_change} days")
print(f"COMPUTED value: {computed} days")

if computed == TRUE_days_since_medication_change:
    print("\nMATCH! Feature implementation is correct.")
else:
    print("\nMISMATCH! There's a bug in the implementation.")

---

## Feature 3: `emergency_visits_last_180d`

**Definition**: Count of emergency room visits in the 180 days before the cutoff date.

**Clinical Relevance**:
- High ER utilization indicates unstable health
- Important predictor of future adverse events

**Pattern Type**: Rolling window (event counting)

### Step 1: Manually Compute TRUE Value

In [ ]:
# STEP 1: Manually compute TRUE value
window_start = CUTOFF_DATE - timedelta(days=180)

# Find emergency encounters
enc_in_window = patient_enc[
    (patient_enc['date'] >= window_start) & 
    (patient_enc['date'] <= CUTOFF_DATE)
]

# Check for emergency encounters using description column
emergency_enc = enc_in_window[
    enc_in_window[enc_desc_col].str.lower().str.contains('emergency', na=False)
]

print(f"180-day window: {window_start.date()} to {CUTOFF_DATE.date()}")
print(f"\nEmergency visits in window:")
for _, row in emergency_enc.iterrows():
    print(f"  {row['date'].date()}: {row[enc_desc_col]}")

TRUE_emergency_visits_last_180d = len(emergency_enc)
print(f"\nTRUE VALUE: {TRUE_emergency_visits_last_180d} visits")

### Step 2: Implement the Function

In [ ]:
# STEP 2: Implement the function
def emergency_visits_last_180d(patient_id, cutoff_date):
    """
    Count emergency visits in 180 days before cutoff.
    
    Args:
        patient_id: Patient identifier
        cutoff_date: Date to calculate from
    
    Returns:
        int: Number of emergency visits
    """
    patient_enc = encounters[encounters[enc_patient_col] == patient_id].copy()
    patient_enc['date'] = pd.to_datetime(patient_enc[enc_date_col], errors='coerce', utc=True)
    
    if cutoff_date.tzinfo is None:
        cutoff_date = pd.to_datetime(cutoff_date, utc=True)
    
    window_start = cutoff_date - timedelta(days=180)
    
    enc_in_window = patient_enc[
        (patient_enc['date'] >= window_start) & 
        (patient_enc['date'] <= cutoff_date)
    ]
    
    emergency_enc = enc_in_window[
        enc_in_window[enc_desc_col].str.lower().str.contains('emergency', na=False)
    ]
    
    return len(emergency_enc)

print("emergency_visits_last_180d function defined")

### Step 3: Verify Against TRUE Value

In [ ]:
# STEP 3: Verify
computed = emergency_visits_last_180d(SAMPLE_PATIENT_ID, CUTOFF_DATE)
print(f"TRUE value:     {TRUE_emergency_visits_last_180d}")
print(f"COMPUTED value: {computed}")

if computed == TRUE_emergency_visits_last_180d:
    print("\nMATCH! Feature implementation is correct.")
else:
    print("\nMISMATCH! There's a bug in the implementation.")

---

## Feature 4: `medication_changes_last_90d`

**Definition**: Count of medication changes (starts or stops) in the 90 days before cutoff.

**Clinical Relevance**:
- Frequent medication changes indicate unstable treatment
- May signal deteriorating condition or side effects

**Pattern Type**: Rolling window (change detection)

### Step 1: Manually Compute TRUE Value

In [ ]:
# STEP 1: Manually compute TRUE value
window_start = CUTOFF_DATE - timedelta(days=90)

med_changes_in_window = []
for _, row in patient_meds.iterrows():
    if pd.notna(row['start_date']) and window_start <= row['start_date'] <= CUTOFF_DATE:
        med_changes_in_window.append(('start', row['start_date']))
    if 'stop_date' in row and pd.notna(row.get('stop_date')):
        stop_date = row['stop_date']
        if window_start <= stop_date <= CUTOFF_DATE:
            med_changes_in_window.append(('stop', stop_date))

print(f"90-day window: {window_start.date()} to {CUTOFF_DATE.date()}")
print(f"\nMedication changes in window:")
for change_type, d in sorted(med_changes_in_window, key=lambda x: x[1]):
    print(f"  {d.date()}: {change_type}")

TRUE_medication_changes_last_90d = len(med_changes_in_window)
print(f"\nTRUE VALUE: {TRUE_medication_changes_last_90d} changes")

### Step 2: Implement the Function

In [ ]:
# STEP 2: Implement the function
def medication_changes_last_90d(patient_id, cutoff_date):
    """
    Count medication changes in 90 days before cutoff.
    
    Args:
        patient_id: Patient identifier
        cutoff_date: Date to calculate from
    
    Returns:
        int: Number of medication changes
    """
    patient_meds = medications[medications[med_patient_col] == patient_id].copy()
    patient_meds['start_date'] = pd.to_datetime(patient_meds[med_start_col], errors='coerce', utc=True)
    
    if cutoff_date.tzinfo is None:
        cutoff_date = pd.to_datetime(cutoff_date, utc=True)
    
    window_start = cutoff_date - timedelta(days=90)
    count = 0
    
    for _, row in patient_meds.iterrows():
        if pd.notna(row['start_date']) and window_start <= row['start_date'] <= cutoff_date:
            count += 1
        if med_stop_col and med_stop_col in patient_meds.columns:
            stop_date = pd.to_datetime(row[med_stop_col], errors='coerce', utc=True)
            if pd.notna(stop_date) and window_start <= stop_date <= cutoff_date:
                count += 1
    
    return count

print("medication_changes_last_90d function defined")

### Step 3: Verify Against TRUE Value

In [ ]:
# STEP 3: Verify
computed = medication_changes_last_90d(SAMPLE_PATIENT_ID, CUTOFF_DATE)
print(f"TRUE value:     {TRUE_medication_changes_last_90d}")
print(f"COMPUTED value: {computed}")

if computed == TRUE_medication_changes_last_90d:
    print("\nMATCH! Feature implementation is correct.")
else:
    print("\nMISMATCH! There's a bug in the implementation.")

---

## Feature 5: `active_medication_count`

**Definition**: Number of medications that are active at the cutoff date (started before cutoff, not stopped or stopped after cutoff).

**Clinical Relevance**:
- Polypharmacy (many medications) increases risk
- Important for drug interaction assessment

**Pattern Type**: Clinical state (active item logic)

### Step 1: Manually Compute TRUE Value

In [ ]:
# STEP 1: Manually compute TRUE value
# A medication is active if: start <= cutoff AND (stop is null OR stop > cutoff)

active_meds = []
for _, row in patient_meds.iterrows():
    start_date = row['start_date']
    stop_date = row.get('stop_date') if 'stop_date' in row else None
    
    if pd.isna(start_date) or start_date > CUTOFF_DATE:
        continue
    
    # Check if still active at cutoff
    if pd.isna(stop_date) or stop_date > CUTOFF_DATE:
        active_meds.append(row)

print(f"Active medications at cutoff ({CUTOFF_DATE.date()}):")
med_desc_col = get_col(medications, 'description')
for row in active_meds[:10]:
    med_name = row[med_desc_col] if med_desc_col else 'Unknown'
    print(f"  {med_name[:50]}...")

TRUE_active_medication_count = len(active_meds)
print(f"\nTRUE VALUE: {TRUE_active_medication_count} medications")

### Step 2: Implement the Function

In [ ]:
# STEP 2: Implement the function
def active_medication_count(patient_id, cutoff_date):
    """
    Count medications active at cutoff date.
    
    A medication is active if: start <= cutoff AND (stop is null OR stop > cutoff)
    
    Args:
        patient_id: Patient identifier
        cutoff_date: Date to check
    
    Returns:
        int: Number of active medications
    """
    patient_meds = medications[medications[med_patient_col] == patient_id].copy()
    patient_meds['start_date'] = pd.to_datetime(patient_meds[med_start_col], errors='coerce', utc=True)
    
    if cutoff_date.tzinfo is None:
        cutoff_date = pd.to_datetime(cutoff_date, utc=True)
    
    count = 0
    for _, row in patient_meds.iterrows():
        start_date = row['start_date']
        
        if pd.isna(start_date) or start_date > cutoff_date:
            continue
        
        stop_date = None
        if med_stop_col and med_stop_col in patient_meds.columns:
            stop_date = pd.to_datetime(row[med_stop_col], errors='coerce', utc=True)
        
        # Active if stop is null or after cutoff
        if pd.isna(stop_date) or stop_date > cutoff_date:
            count += 1
    
    return count

print("active_medication_count function defined")

### Step 3: Verify Against TRUE Value

In [ ]:
# STEP 3: Verify
computed = active_medication_count(SAMPLE_PATIENT_ID, CUTOFF_DATE)
print(f"TRUE value:     {TRUE_active_medication_count}")
print(f"COMPUTED value: {computed}")

if computed == TRUE_active_medication_count:
    print("\nMATCH! Feature implementation is correct.")
else:
    print("\nMISMATCH! There's a bug in the implementation.")

---

## Feature 6: `hba1c_trend_last_180d`

**Definition**: Direction of HbA1c change over the last 180 days: -1 (improving/decreasing), 0 (stable), +1 (worsening/increasing).

**Clinical Relevance**:
- Rising HbA1c indicates worsening glycemic control
- Falling HbA1c indicates treatment success
- Threshold: ±0.5% change is considered significant

**Pattern Type**: Trend (direction detection)

### Step 1: Manually Compute TRUE Value

In [ ]:
# STEP 1: Manually compute TRUE value
TREND_THRESHOLD = 0.5  # Change >= 0.5% is significant
window_start = CUTOFF_DATE - timedelta(days=180)

hba1c_in_window = patient_obs[
    (patient_obs['date'] >= window_start) & 
    (patient_obs['date'] <= CUTOFF_DATE) &
    (patient_obs[obs_desc_col].str.contains('hemoglobin a1c|hba1c', case=False, na=False))
].sort_values('date')

print(f"HbA1c readings in 180-day window:")
for _, row in hba1c_in_window.iterrows():
    print(f"  {row['date'].date()}: {row[obs_value_col]}%")

if len(hba1c_in_window) >= 2:
    first_value = float(hba1c_in_window.iloc[0][obs_value_col])
    last_value = float(hba1c_in_window.iloc[-1][obs_value_col])
    change = last_value - first_value
    
    if change >= TREND_THRESHOLD:
        TRUE_hba1c_trend_last_180d = 1  # Worsening
    elif change <= -TREND_THRESHOLD:
        TRUE_hba1c_trend_last_180d = -1  # Improving
    else:
        TRUE_hba1c_trend_last_180d = 0  # Stable
    
    print(f"\nFirst: {first_value}%, Last: {last_value}%, Change: {change:+.1f}%")
else:
    TRUE_hba1c_trend_last_180d = 0  # Not enough data = stable
    print("\nNot enough readings for trend calculation")

print(f"TRUE VALUE: {TRUE_hba1c_trend_last_180d} (-1=improving, 0=stable, 1=worsening)")

### Step 2: Implement the Function

In [ ]:
# STEP 2: Implement the function
def hba1c_trend_last_180d(patient_id, cutoff_date, threshold=0.5):
    """
    HbA1c trend direction in last 180 days.
    
    Args:
        patient_id: Patient identifier
        cutoff_date: Date to calculate from
        threshold: Change threshold for trend (default 0.5%)
    
    Returns:
        int: -1 (improving), 0 (stable), or 1 (worsening)
    """
    patient_obs = observations[observations[obs_patient_col] == patient_id].copy()
    patient_obs['date'] = pd.to_datetime(patient_obs[obs_date_col], utc=True)
    
    if cutoff_date.tzinfo is None:
        cutoff_date = pd.to_datetime(cutoff_date, utc=True)
    
    window_start = cutoff_date - timedelta(days=180)
    
    hba1c_in_window = patient_obs[
        (patient_obs['date'] >= window_start) & 
        (patient_obs['date'] <= cutoff_date) &
        (patient_obs[obs_desc_col].str.contains('hemoglobin a1c|hba1c', case=False, na=False))
    ].sort_values('date')
    
    if len(hba1c_in_window) < 2:
        return 0  # Not enough data
    
    first_value = float(hba1c_in_window.iloc[0][obs_value_col])
    last_value = float(hba1c_in_window.iloc[-1][obs_value_col])
    change = last_value - first_value
    
    if change >= threshold:
        return 1  # Worsening
    elif change <= -threshold:
        return -1  # Improving
    else:
        return 0  # Stable

print("hba1c_trend_last_180d function defined")

### Step 3: Verify Against TRUE Value

In [ ]:
# STEP 3: Verify
computed = hba1c_trend_last_180d(SAMPLE_PATIENT_ID, CUTOFF_DATE)
print(f"TRUE value:     {TRUE_hba1c_trend_last_180d}")
print(f"COMPUTED value: {computed}")

if computed == TRUE_hba1c_trend_last_180d:
    print("\nMATCH! Feature implementation is correct.")
else:
    print("\nMISMATCH! There's a bug in the implementation.")

---

## Feature 7: `longest_care_gap_days`

**Definition**: The longest gap (in days) between consecutive HbA1c tests before the cutoff date.

**Clinical Relevance**:
- Long gaps in monitoring indicate care engagement issues
- ADA recommends HbA1c every 3-6 months

**Pattern Type**: Gap detection (max of consecutive gaps)

### Step 1: Manually Compute TRUE Value

In [ ]:
# STEP 1: Manually compute TRUE value
hba1c_before = patient_obs[
    (patient_obs['date'] <= CUTOFF_DATE) &
    (patient_obs[obs_desc_col].str.contains('hemoglobin a1c|hba1c', case=False, na=False))
].sort_values('date')

print(f"HbA1c readings BEFORE cutoff:")
for _, row in hba1c_before.iterrows():
    print(f"  {row['date'].date()}")

if len(hba1c_before) >= 2:
    dates = hba1c_before['date'].tolist()
    gaps = [(dates[i+1] - dates[i]).days for i in range(len(dates)-1)]
    
    print(f"\nGaps between tests:")
    for i, gap in enumerate(gaps):
        print(f"  {dates[i].date()} -> {dates[i+1].date()}: {gap} days")
    
    TRUE_longest_care_gap_days = max(gaps)
else:
    TRUE_longest_care_gap_days = None
    
print(f"\nTRUE VALUE: {TRUE_longest_care_gap_days} days")

### Step 2: Implement the Function

In [ ]:
# STEP 2: Implement the function
def longest_care_gap_days(patient_id, cutoff_date):
    """
    Longest gap between consecutive HbA1c tests before cutoff.
    
    Args:
        patient_id: Patient identifier
        cutoff_date: Date to calculate from
    
    Returns:
        int: Maximum gap in days, or None if < 2 tests
    """
    patient_obs = observations[observations[obs_patient_col] == patient_id].copy()
    patient_obs['date'] = pd.to_datetime(patient_obs[obs_date_col], utc=True)
    
    hba1c_before = patient_obs[
        (patient_obs['date'] <= cutoff_date) &
        (patient_obs[obs_desc_col].str.contains('hemoglobin a1c|hba1c', case=False, na=False))
    ].sort_values('date')
    
    if len(hba1c_before) < 2:
        return None
    
    dates = hba1c_before['date'].tolist()
    gaps = [(dates[i+1] - dates[i]).days for i in range(len(dates)-1)]
    
    return max(gaps)

print("longest_care_gap_days function defined")

### Step 3: Verify Against TRUE Value

In [ ]:
# STEP 3: Verify
computed = longest_care_gap_days(SAMPLE_PATIENT_ID, CUTOFF_DATE)
print(f"TRUE value:     {TRUE_longest_care_gap_days} days")
print(f"COMPUTED value: {computed} days")

if computed == TRUE_longest_care_gap_days:
    print("\nMATCH! Feature implementation is correct.")
else:
    print("\nMISMATCH! There's a bug in the implementation.")

---

## Feature 8: `bmi_category`

**Definition**: BMI category at the cutoff date based on most recent height and weight.

Categories:
- 1 = Underweight (BMI < 18.5)
- 2 = Normal (18.5 <= BMI < 25)
- 3 = Overweight (25 <= BMI < 30)
- 4 = Obese (BMI >= 30)

**Clinical Relevance**:
- Obesity is a major risk factor for diabetes complications
- Weight management is key to glycemic control

**Pattern Type**: Categorical (BMI calculation and binning)

### Step 1: Manually Compute TRUE Value

In [ ]:
# STEP 1: Manually compute TRUE value
# Find most recent height and weight before cutoff

# Height (Body Height) - LOINC 8302-2
height_obs = patient_obs[
    (patient_obs['date'] <= CUTOFF_DATE) &
    (patient_obs[obs_desc_col].str.lower().str.contains('body height|height', na=False))
].sort_values('date')

# Weight (Body Weight) - LOINC 29463-7
weight_obs = patient_obs[
    (patient_obs['date'] <= CUTOFF_DATE) &
    (patient_obs[obs_desc_col].str.lower().str.contains('body weight|weight', na=False))
].sort_values('date')

print(f"Recent height readings:")
for _, row in height_obs.tail(3).iterrows():
    print(f"  {row['date'].date()}: {row[obs_value_col]}")

print(f"\nRecent weight readings:")
for _, row in weight_obs.tail(3).iterrows():
    print(f"  {row['date'].date()}: {row[obs_value_col]}")

if len(height_obs) > 0 and len(weight_obs) > 0:
    height_cm = float(height_obs.iloc[-1][obs_value_col])
    weight_kg = float(weight_obs.iloc[-1][obs_value_col])
    height_m = height_cm / 100
    bmi = weight_kg / (height_m ** 2)
    
    if bmi < 18.5:
        TRUE_bmi_category = 1  # Underweight
    elif bmi < 25:
        TRUE_bmi_category = 2  # Normal
    elif bmi < 30:
        TRUE_bmi_category = 3  # Overweight
    else:
        TRUE_bmi_category = 4  # Obese
    
    print(f"\nHeight: {height_cm} cm, Weight: {weight_kg} kg")
    print(f"BMI: {bmi:.1f}")
else:
    TRUE_bmi_category = None
    print("\nMissing height or weight data")
    
print(f"\nTRUE VALUE: {TRUE_bmi_category} (1=underweight, 2=normal, 3=overweight, 4=obese)")

### Step 2: Implement the Function

In [ ]:
# STEP 2: Implement the function
def bmi_category(patient_id, cutoff_date):
    """
    BMI category based on most recent height and weight.
    
    Categories:
        1 = Underweight (BMI < 18.5)
        2 = Normal (18.5 <= BMI < 25)
        3 = Overweight (25 <= BMI < 30)
        4 = Obese (BMI >= 30)
    
    Args:
        patient_id: Patient identifier
        cutoff_date: Date to calculate from
    
    Returns:
        int: BMI category (1-4), or None if missing data
    """
    patient_obs = observations[observations[obs_patient_col] == patient_id].copy()
    patient_obs['date'] = pd.to_datetime(patient_obs[obs_date_col], utc=True)
    
    # Find most recent height
    height_obs = patient_obs[
        (patient_obs['date'] <= cutoff_date) &
        (patient_obs[obs_desc_col].str.lower().str.contains('body height|height', na=False))
    ].sort_values('date')
    
    # Find most recent weight
    weight_obs = patient_obs[
        (patient_obs['date'] <= cutoff_date) &
        (patient_obs[obs_desc_col].str.lower().str.contains('body weight|weight', na=False))
    ].sort_values('date')
    
    if len(height_obs) == 0 or len(weight_obs) == 0:
        return None
    
    height_cm = float(height_obs.iloc[-1][obs_value_col])
    weight_kg = float(weight_obs.iloc[-1][obs_value_col])
    height_m = height_cm / 100
    bmi = weight_kg / (height_m ** 2)
    
    if bmi < 18.5:
        return 1  # Underweight
    elif bmi < 25:
        return 2  # Normal
    elif bmi < 30:
        return 3  # Overweight
    else:
        return 4  # Obese

print("bmi_category function defined")

### Step 3: Verify Against TRUE Value

In [ ]:
# STEP 3: Verify
computed = bmi_category(SAMPLE_PATIENT_ID, CUTOFF_DATE)
print(f"TRUE value:     {TRUE_bmi_category}")
print(f"COMPUTED value: {computed}")

if computed == TRUE_bmi_category:
    print("\nMATCH! Feature implementation is correct.")
else:
    print("\nMISMATCH! There's a bug in the implementation.")

---

# Part 2: Features Provided to You (8 features)

These features are provided with complete implementations. Study them to understand the patterns.

In [ ]:
# PROVIDED FEATURE 1: days_since_emergency_visit
def days_since_emergency_visit(patient_id, cutoff_date):
    """Days since most recent emergency visit before cutoff."""
    patient_enc = encounters[encounters[enc_patient_col] == patient_id].copy()
    patient_enc['date'] = pd.to_datetime(patient_enc[enc_date_col], errors='coerce', utc=True)
    
    emergency_before = patient_enc[
        (patient_enc['date'] <= cutoff_date) &
        (patient_enc[enc_desc_col].str.lower().str.contains('emergency', na=False))
    ]
    
    if len(emergency_before) == 0:
        return None
    
    last_date = emergency_before['date'].max()
    return (cutoff_date - last_date).days

# PROVIDED FEATURE 2: days_since_hospitalization
def days_since_hospitalization(patient_id, cutoff_date):
    """Days since most recent hospitalization before cutoff."""
    patient_enc = encounters[encounters[enc_patient_col] == patient_id].copy()
    patient_enc['date'] = pd.to_datetime(patient_enc[enc_date_col], errors='coerce', utc=True)
    
    hosp_before = patient_enc[
        (patient_enc['date'] <= cutoff_date) &
        (patient_enc[enc_desc_col].str.lower().str.contains('inpatient|admission to|hospital admission', na=False))
    ]
    
    if len(hosp_before) == 0:
        return None
    
    last_date = hosp_before['date'].max()
    return (cutoff_date - last_date).days

# PROVIDED FEATURE 3: hospitalizations_last_365d
def hospitalizations_last_365d(patient_id, cutoff_date):
    """Count hospitalizations in 365 days before cutoff."""
    patient_enc = encounters[encounters[enc_patient_col] == patient_id].copy()
    patient_enc['date'] = pd.to_datetime(patient_enc[enc_date_col], errors='coerce', utc=True)
    
    if cutoff_date.tzinfo is None:
        cutoff_date = pd.to_datetime(cutoff_date, utc=True)
    
    window_start = cutoff_date - timedelta(days=365)
    
    hosp_in_window = patient_enc[
        (patient_enc['date'] >= window_start) & 
        (patient_enc['date'] <= cutoff_date) &
        (patient_enc[enc_desc_col].str.lower().str.contains('inpatient|admission to|hospital admission', na=False))
    ]
    
    return len(hosp_in_window)
print("Provided features 1-3 defined")

In [ ]:
# PROVIDED FEATURE 4: diabetes_complication_count
DIABETES_COMPLICATIONS = [
    'retinopathy', 'nephropathy', 'neuropathy', 'foot ulcer',
    'diabetic ketoacidosis', 'hyperosmolar', 'chronic kidney',
    'peripheral vascular', 'coronary artery', 'stroke'
]

def diabetes_complication_count(patient_id, cutoff_date):
    """Count unique diabetes complications diagnosed before cutoff."""
    patient_cond = conditions[conditions[cond_patient_col] == patient_id].copy()
    
    # Get condition start dates if available
    cond_start_col = get_col(conditions, 'start')
    if cond_start_col:
        patient_cond['start_date'] = pd.to_datetime(patient_cond[cond_start_col], errors='coerce', utc=True)
        patient_cond = patient_cond[patient_cond['start_date'] <= cutoff_date]
    
    complications = set()
    for _, row in patient_cond.iterrows():
        desc = str(row[cond_desc_col]).lower()
        for comp in DIABETES_COMPLICATIONS:
            if comp in desc:
                complications.add(comp)
    
    return len(complications)

# PROVIDED FEATURE 5: care_gaps_count
def care_gaps_count(patient_id, cutoff_date, gap_threshold=180):
    """Count HbA1c test gaps > threshold days before cutoff."""
    patient_obs = observations[observations[obs_patient_col] == patient_id].copy()
    patient_obs['date'] = pd.to_datetime(patient_obs[obs_date_col], utc=True)
    
    hba1c_before = patient_obs[
        (patient_obs['date'] <= cutoff_date) &
        (patient_obs[obs_desc_col].str.contains('hemoglobin a1c|hba1c', case=False, na=False))
    ].sort_values('date')
    
    if len(hba1c_before) < 2:
        return 0
    
    dates = hba1c_before['date'].tolist()
    gaps = [(dates[i+1] - dates[i]).days for i in range(len(dates)-1)]
    
    return sum(1 for g in gaps if g > gap_threshold)

print("Provided features 4-5 defined")

In [ ]:
# PROVIDED FEATURE 6: current_diastolic_bp (LOINC 8462-4)
def current_diastolic_bp(patient_id, cutoff_date):
    """Most recent diastolic BP value before cutoff. LOINC: 8462-4"""
    LOINC_DIASTOLIC_BP = '8462-4'
    
    patient_obs = observations[observations[obs_patient_col] == patient_id].copy()
    patient_obs['date'] = pd.to_datetime(patient_obs[obs_date_col], utc=True)
    
    bp_before = patient_obs[
        (patient_obs['date'] <= cutoff_date) &
        ((patient_obs[obs_code_col] == LOINC_DIASTOLIC_BP) |
         (patient_obs[obs_desc_col].str.lower().str.contains('diastolic', na=False)))
    ]
    
    if len(bp_before) == 0:
        return None
    
    last_value = bp_before.loc[bp_before['date'].idxmax(), obs_value_col]
    return float(last_value)

# PROVIDED FEATURE 7: bp_trend_last_180d
def bp_trend_last_180d(patient_id, cutoff_date, threshold=10):
    """Blood pressure trend direction in last 180 days.
    
    Returns: -1 (improving/decreasing), 0 (stable), 1 (worsening/increasing)
    Threshold: ±10 mmHg change is considered significant.
    """
    LOINC_SYSTOLIC_BP = '8480-6'
    
    patient_obs = observations[observations[obs_patient_col] == patient_id].copy()
    patient_obs['date'] = pd.to_datetime(patient_obs[obs_date_col], utc=True)
    
    if cutoff_date.tzinfo is None:
        cutoff_date = pd.to_datetime(cutoff_date, utc=True)
    
    window_start = cutoff_date - timedelta(days=180)
    
    bp_in_window = patient_obs[
        (patient_obs['date'] >= window_start) & 
        (patient_obs['date'] <= cutoff_date) &
        ((patient_obs[obs_code_col] == LOINC_SYSTOLIC_BP) |
         (patient_obs[obs_desc_col].str.lower().str.contains('systolic', na=False)))
    ].sort_values('date')
    
    if len(bp_in_window) < 2:
        return 0
    
    first_value = float(bp_in_window.iloc[0][obs_value_col])
    last_value = float(bp_in_window.iloc[-1][obs_value_col])
    change = last_value - first_value
    
    if change >= threshold:
        return 1  # Worsening (BP increasing)
    elif change <= -threshold:
        return -1  # Improving (BP decreasing)
    else:
        return 0  # Stable

# PROVIDED FEATURE 8: egfr_trend_last_365d
def egfr_trend_last_365d(patient_id, cutoff_date, threshold=5):
    """eGFR trend direction in last 365 days.
    
    Returns: -1 (worsening/decreasing), 0 (stable), 1 (improving/increasing)
    Note: For eGFR, DECREASING is bad (kidney function declining).
    Threshold: ±5 mL/min change is considered significant.
    """
    LOINC_EGFR = '33914-3'
    
    patient_obs = observations[observations[obs_patient_col] == patient_id].copy()
    patient_obs['date'] = pd.to_datetime(patient_obs[obs_date_col], utc=True)
    
    if cutoff_date.tzinfo is None:
        cutoff_date = pd.to_datetime(cutoff_date, utc=True)
    
    window_start = cutoff_date - timedelta(days=365)
    
    egfr_in_window = patient_obs[
        (patient_obs['date'] >= window_start) & 
        (patient_obs['date'] <= cutoff_date) &
        ((patient_obs[obs_code_col] == LOINC_EGFR) |
         (patient_obs[obs_desc_col].str.lower().str.contains('glomerular|egfr', na=False)))
    ].sort_values('date')
    
    if len(egfr_in_window) < 2:
        return 0
    
    first_value = float(egfr_in_window.iloc[0][obs_value_col])
    last_value = float(egfr_in_window.iloc[-1][obs_value_col])
    change = last_value - first_value
    
    if change <= -threshold:
        return -1  # Worsening (eGFR decreasing)
    elif change >= threshold:
        return 1  # Improving (eGFR increasing)
    else:
        return 0  # Stable

print("Provided features 6-8 defined")

---

## Feature Summary

Let's verify all features work correctly.

In [ ]:
# Test all features on sample patient
ALL_FEATURES = {
    # Session 1 features (6)
    'days_since_last_hba1c': days_since_last_hba1c,
    'current_hba1c_level': current_hba1c_level,
    'encounters_last_90d': encounters_last_90d,
    'age_at_date': age_at_date,
    'current_systolic_bp': current_systolic_bp,
    'current_egfr': current_egfr,
    
    # Your features (8)
    'days_since_last_encounter': days_since_last_encounter,
    'days_since_medication_change': days_since_medication_change,
    'emergency_visits_last_180d': emergency_visits_last_180d,
    'medication_changes_last_90d': medication_changes_last_90d,
    'active_medication_count': active_medication_count,
    'hba1c_trend_last_180d': hba1c_trend_last_180d,
    'longest_care_gap_days': longest_care_gap_days,
    'bmi_category': bmi_category,
    
    # Provided features (8)
    'days_since_emergency_visit': days_since_emergency_visit,
    'days_since_hospitalization': days_since_hospitalization,
    'hospitalizations_last_365d': hospitalizations_last_365d,
    'diabetes_complication_count': diabetes_complication_count,
    'care_gaps_count': care_gaps_count,
    'current_diastolic_bp': current_diastolic_bp,
    'bp_trend_last_180d': bp_trend_last_180d,
    'egfr_trend_last_365d': egfr_trend_last_365d,
}

print(f"Testing all {len(ALL_FEATURES)} features on sample patient...")
print(f"Patient: {SAMPLE_PATIENT_ID}")
print(f"Cutoff: {CUTOFF_DATE.date()}")
print("=" * 60)

for name, func in ALL_FEATURES.items():
    try:
        value = func(SAMPLE_PATIENT_ID, CUTOFF_DATE)
        print(f"{name:<35} = {value}")
    except Exception as e:
        print(f"{name:<35} ERROR: {e}")

print(f"\nTotal features: {len(ALL_FEATURES)}")

### TDD Quality Check

A good TDD test uses **non-trivial TRUE values**. If a feature's TRUE value is 0 or None, the test cannot distinguish a correct implementation from one that simply returns a constant. Let's check which features produced meaningful test values.

In [ ]:
# Check which features have trivial (0 or None) test values
trivial_features = []
nontrivial_features = []

for name, func in ALL_FEATURES.items():
    value = func(SAMPLE_PATIENT_ID, CUTOFF_DATE)
    if value is None or value == 0:
        trivial_features.append((name, value))
    else:
        nontrivial_features.append((name, value))

print(f"Non-trivial test values ({len(nontrivial_features)}/{len(ALL_FEATURES)}):")
for name, val in nontrivial_features:
    print(f"  {name:<35} = {val}")

if trivial_features:
    print(f"\nTrivial test values ({len(trivial_features)}/{len(ALL_FEATURES)}):")
    for name, val in trivial_features:
        print(f"  {name:<35} = {val}")
    print(f"\nThese features passed TDD but the test is weak (0/None matches too easily).")
    print("Consider verifying these with a different patient or cutoff date.")
else:
    print("\nAll features have non-trivial test values. TDD coverage is strong.")

---

# Part 3: Transfer Your Functions to ExtendedPatientProfile

In this homework you implemented 8 features as **standalone notebook functions** that query raw DataFrames each time they run. Now you'll transfer them into `ExtendedPatientProfile` as **class methods**.

### Why class methods?

| | Notebook functions | Class methods |
|---|---|---|
| **Data access** | Filter raw DataFrames every call | Use pre-loaded `self.*_timeline` data |
| **Helper methods** | Must rewrite common patterns | Reuse `_days_since_last_event()`, `_count_events_in_window()`, etc. |
| **Speed** | Slow (re-reads data each time) | Fast (data loaded once in `__init__`) |
| **Integration** | Standalone | Plugs into `get_daily_features()` for batch generation |

### What to do

1. Open `src/profiles/extended_patient_profile.py`
2. Find the section: **HOMEWORK TODO: Implement these 8 feature methods**
3. Each TODO method corresponds to one of your notebook functions
4. Adapt the logic to use `self.*` attributes and helper methods instead of raw DataFrames

**Example** — your notebook function:
```python
def days_since_last_encounter(patient_id, cutoff_date):
    patient_enc = encounters[encounters[enc_patient_col] == patient_id]
    ...
```

becomes a class method:
```python
def _days_since_last_encounter(self, target_date):
    return self._days_since_last_event(target_date, self.encounter_timeline)
```

The class already has the data loaded (`self.encounter_timeline`, `self.hba1c_timeline`, etc.) and helper methods (`_days_since_last_event`, `_count_events_in_window`) that handle the common patterns.

In [ ]:
# Re-install the src package to pick up your changes
!pip install {REPO_PATH}/src_solutions/ -q

# Reload the module (needed because Python caches imports)
import importlib
import week_2.profiles.extended_patient_profile as epp_module
importlib.reload(epp_module)
from week_2.profiles.extended_patient_profile import ExtendedPatientProfile
print("ExtendedPatientProfile reloaded")

In [ ]:
# Load profiles for verification
profiles = ExtendedPatientProfile.load_from_dataframes(
    patients_df=patients, encounters_df=encounters,
    conditions_df=conditions, observations_df=observations,
    medications_df=medications, diabetic_only=True, show_progress=True
)

# Verify your class method implementations
# Note: ExtendedPatientProfile uses tz-naive dates internally,
# so we pass a date string instead of the tz-aware CUTOFF_DATE.
test_profile = profiles[SAMPLE_PATIENT_ID]
test_date = CUTOFF_DATE.strftime('%Y-%m-%d')

hw_features = test_profile.get_homework_features(test_date)

print("Homework features from ExtendedPatientProfile class methods:")
print("=" * 60)
implemented = 0
for name, value in hw_features.items():
    status = "STUB" if value == -999 else "OK"
    if value != -999:
        implemented += 1
    print(f"  {name:<35} = {str(value):<10} [{status}]")

print(f"\nImplemented: {implemented}/{len(hw_features)}")
if implemented == len(hw_features):
    print("All homework methods implemented!")
else:
    print(f"{len(hw_features) - implemented} methods still have TODO stubs.")
    print("Edit extended_patient_profile.py, re-run the pip install cell above,")
    print("then re-run this cell.")

---

# Part 4: Generate Classifier Training Data

Use ExtendedPatientProfile to generate the complete training dataset with all 22 features.

In [ ]:
# Import ExtendedPatientProfile
try:
    import tqdm
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'tqdm', '-q'])

from week_2.profiles.extended_patient_profile import ExtendedPatientProfile

print("ExtendedPatientProfile imported")

In [ ]:
# Classify encounters again (if needed)
if 'ENCOUNTER_CLASS' not in encounters.columns:
    encounters['ENCOUNTER_CLASS'] = encounters[enc_desc_col].apply(classify_encounter)
    print("Encounter classification complete")
else:
    print("Encounters already classified")

In [ ]:
# Profiles already loaded in Part 3 verification cell above
print(f"Using {len(profiles):,} profiles")


In [ ]:
# Generate training data with all features (interval_days=7)
from tqdm import tqdm

print("Generating classifier training data with all 22 features...")
print("Using interval_days=7 for practical dataset size")

all_instances = []
for profile in tqdm(profiles.values(), desc="Processing patients"):
    instances = profile.generate_all_instances(interval_days=7)
    all_instances.extend(instances)

classifier_df = pd.DataFrame(all_instances)
print(f"\nGenerated {len(classifier_df):,} instances")
print(f"Columns: {list(classifier_df.columns)}")

> **Note:** The generated dataset includes two additional columns (`survival_days_until_event`, `survival_event_observed`) for optional survival analysis in later sessions. These are excluded from the feature set for binary classification and will not affect our models.

In [ ]:
# Save updated classifier data
OUTPUT_PATH = os.path.join(REPO_PATH, "data/week_2", "classifier_training_data_22_features.csv")
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
classifier_df.to_csv(OUTPUT_PATH, index=False)

feature_cols = [c for c in classifier_df.columns 
                if c not in ['patient_id', 'date', 'will_have_high_risk_event_next_30d', 
                            'survival_days_until_event', 'survival_event_observed']]

print(f"Saved to {OUTPUT_PATH}")
print(f"\nFeatures ({len(feature_cols)}):")
for f in feature_cols:
    print(f"  - {f}")

---

# Part 5: Compare Baselines (6 vs 22 Features)

Compare model performance using 6 Session 1 features vs all 22 features.

We use the same 22-feature dataset for both comparisons (just with different feature subsets) to ensure a fair comparison on identical instances. We also apply `StandardScaler` to normalize features before fitting logistic regression -- this prevents convergence issues when features are on different scales (e.g., age 12-96 vs encounters 0-12).

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.preprocessing import StandardScaler

# Use the 22-feature dataset for both comparisons
# (same instances, different feature subsets -- ensures fair comparison)
data_22 = classifier_df.copy()

label_col = 'will_have_high_risk_event_next_30d'
patient_col = 'patient_id'

# Feature lists
features_6 = ['days_since_last_hba1c', 'current_hba1c_level', 'encounters_last_90d',
              'age_at_date', 'current_systolic_bp', 'current_egfr']

features_22 = [c for c in data_22.columns 
               if c not in ['patient_id', 'date', label_col, 
                           'survival_days_until_event', 'survival_event_observed']]

print(f"6-feature model: {features_6}")
print(f"\n22-feature model ({len(features_22)} features): {features_22}")

In [ ]:
def evaluate_features(data, feature_cols, name):
    """Train and evaluate a model with given features."""
    data = data.dropna(subset=[label_col])
    
    X = data[feature_cols].fillna(0)
    y = data[label_col].astype(int)
    groups = data[patient_col]
    
    # Patient-level split
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    for train_idx, test_idx in gss.split(X, y, groups):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    # Standardize features (prevents convergence issues with different scales)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Train
    model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
    model.fit(X_train_scaled, y_train)
    
    # Evaluate
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    roc_auc = roc_auc_score(y_test, y_prob)
    pr_auc = average_precision_score(y_test, y_prob)
    
    print(f"\n{name}:")
    print(f"  ROC-AUC: {roc_auc:.4f}")
    print(f"  PR-AUC:  {pr_auc:.4f}")
    
    return roc_auc, pr_auc, model, scaler

# Evaluate both
print("=" * 50)
print("BASELINE COMPARISON")
print("=" * 50)

roc_6, pr_6, model_6, scaler_6 = evaluate_features(data_22, features_6, "6 Features (Session 1)")
roc_22, pr_22, model_22, scaler_22 = evaluate_features(data_22, features_22, "22 Features (Complete)")

print("\n" + "=" * 50)
print(f"PR-AUC Improvement: {(pr_22 - pr_6) / pr_6 * 100:.1f}%")

In [ ]:
import joblib

MODEL_DIR = os.path.join(REPO_PATH, "models/week_2")
os.makedirs(MODEL_DIR, exist_ok=True)

# Save models and their scalers
joblib.dump(model_6, os.path.join(MODEL_DIR, "6_feature_logistic_regression.joblib"))
joblib.dump(scaler_6, os.path.join(MODEL_DIR, "6_feature_scaler.joblib"))
joblib.dump(model_22, os.path.join(MODEL_DIR, "22_feature_logistic_regression.joblib"))
joblib.dump(scaler_22, os.path.join(MODEL_DIR, "22_feature_scaler.joblib"))

print(f"Models saved to {MODEL_DIR}/")
for f in sorted(os.listdir(MODEL_DIR)):
    print(f"  {f}")

---

## Homework Complete!

### What You Accomplished

1. Implemented **8 features** using test-driven development:
   - `days_since_last_encounter`
   - `days_since_medication_change`
   - `emergency_visits_last_180d`
   - `medication_changes_last_90d`
   - `active_medication_count`
   - `hba1c_trend_last_180d`
   - `longest_care_gap_days`
   - `bmi_category`

2. Used **8 provided features** to complete the feature set

3. Transferred notebook functions to **ExtendedPatientProfile class methods**

4. Generated **classifier training data** with all 22 features

5. Compared **6-feature vs 22-feature** baseline performance and saved models

### Key Observations

- Adding 16 features improved PR-AUC, demonstrating that additional clinical signals provide predictive value beyond the initial 6 features.
- `StandardScaler` prevents convergence issues and makes coefficient magnitudes comparable across features.
- The TDD quality check identifies features where the test was weak (TRUE value = 0 or None). In practice, you should verify these with additional test cases.

### Professional Tip

The TDD approach (manual TRUE value, implement, verify) catches bugs early and builds confidence in your features. In production systems, these test cases become automated unit tests that run on every code change.